# Stage 2: Instruction Fine-Tuning
## Finance FAQ Assistant — Teaching the Model to Answer Questions

This notebook:
1. Tests the **base model** on 10 finance questions (Step 5 of the assignment)
2. Fine-tunes on the 104-example instruction dataset, continuing from the Stage 1 adapter
3. Compares base vs. instruction-tuned answers on the same 10 questions (Step 7)

**Runtime:** Google Colab, T4 GPU

In [1]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" trl peft accelerate bitsandbytes
!pip install datasets

## 1. Define the 10 evaluation questions
These same 10 questions are reused in Step 5 (base model), Step 7 (SFT comparison), and Step 10 (final 3-way comparison) so results are directly comparable.

In [2]:
EVAL_QUESTIONS = [
    "How can I apply for a personal loan?",
    "What is the difference between a credit card and a debit card?",
    "What happens if I only pay the minimum amount due on my credit card?",
    "What is a SIP?",
    "What is the ideal credit utilization ratio?",
    "What is the difference between fixed and floating interest rates?",
    "What documents do I need to open a savings account?",
    "What happens if I default on a secured loan?",
    "How can I improve my credit score?",
    "What is the difference between TDS and income tax?",
]
assert len(EVAL_QUESTIONS) == 10

## 2. Load the BASE model (no fine-tuning) and test it — Step 5

In [3]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 512

base_model, base_tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-0.5B-Instruct",  # using the instruct base as the "untrained" reference point
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(base_model)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_pil_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.auto.image_processing_auto`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_beit`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_pil_beit`. R

🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.7.2: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896, padding_idx=151654)
    (layers): ModuleList(
      (0): Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
      (1): Qwe

In [4]:
def ask_base(question, model, tokenizer, max_new_tokens=120):
    messages = [{"role": "user", "content": question}]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")
    out = model.generate(input_ids=inputs, max_new_tokens=max_new_tokens,
                          do_sample=False, temperature=0.1)
    decoded = tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)
    return decoded.strip()

base_answers = {}
for q in EVAL_QUESTIONS:
    ans = ask_base(q, base_model, base_tokenizer)
    base_answers[q] = ans
    print(f"Q: {q}\nBASE: {ans}\n" + "-"*80)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=120) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_

Q: How can I apply for a personal loan?
BASE: I'm sorry, but as an AI language model, I am not able to provide specific guidance on applying for a personal loan. However, here are some general steps you can follow:

1. Research: Before applying for a personal loan, it's important to research the different types of loans available and understand what they entail.

2. Determine your financial needs: Consider how much money you need to borrow and what purpose it will serve in your life.

3. Assess your creditworthiness: Your credit score is one of the most important factors when determining whether or not to lend to you. A good
--------------------------------------------------------------------------------


Both `max_new_tokens` (=120) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: What is the difference between a credit card and a debit card?
BASE: A credit card and a debit card are two types of cards that are used to make purchases or withdraw money from banks.

A credit card is a type of card that allows you to borrow money from your bank account without having to pay interest on the amount borrowed. Credit cards are typically issued by banks and are used for a variety of purposes, including shopping, dining out, and travel.

On the other hand, a debit card is a type of card that does not allow you to borrow money from your bank account. Debit cards are typically used for transactions such as buying groceries at a store, paying
--------------------------------------------------------------------------------


Both `max_new_tokens` (=120) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

Q: What documents do I need to open a savings account?
BASE: Opening a savings account is an important step in establishing your financial security. Here are the basic steps you should follow:

1. **Choose Your Bank**: Decide on the bank that will be used for your savings account. This could be a traditional bank or a high-net-worth individual (HNWI) bank.

2. **Open a Savings Account**: Once you have chosen your bank, you can proceed with opening a savings account. This typically involves providing personal identification such as a driver's license, passport, or other forms of identification.

3. **Fill Out Application Forms**: Depending on your bank,
--------------------------------------------------------------------------------


Both `max_new_tokens` (=120) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: What happens if I default on a secured loan?
BASE: I'm sorry to hear that you're having trouble with secured loans. Secured loans are typically secured by collateral, which means the lender has a claim on your property or assets as security for the loan amount.

If you default on a secured loan, it can have several consequences:

1. **Loss of Loan**: If you fail to repay the loan, the lender will lose possession of the collateral (the property or asset) and may take legal action against you.
2. **Credit Rating Impact**: A negative credit rating could impact your ability to secure future loans in the future.
3. **Legal
--------------------------------------------------------------------------------


Both `max_new_tokens` (=120) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: How can I improve my credit score?
BASE: Improving your credit score is an important step to building a strong financial foundation for your future. Here are some steps you can take:

1. Pay on time: One of the most basic things you can do to improve your credit score is to pay on time for all your debts. This includes paying your bills in full each month and avoiding late payments.

2. Create a budget: Creating a budget that allocates a portion of your income towards debt repayment can also help improve your credit score. A good rule of thumb is to allocate 30% of your income towards debt repayment, 4
--------------------------------------------------------------------------------
Q: What is the difference between TDS and income tax?
BASE: I'm sorry for any confusion caused earlier. I am Qwen, an artificial intelligence designed to provide information and answer questions based on my training data. In this case, there seems to be some misunderstanding about the concept of income ta

In [5]:
import json
with open("base_model_answers.json", "w") as f:
    json.dump(base_answers, f, indent=2)
print("Saved base_model_answers.json — used to build reports/base_model_evaluation.md")

Saved base_model_answers.json — used to build reports/base_model_evaluation.md


## 3. Load Stage 1 adapter (or base model) and prepare for instruction fine-tuning

In [23]:
%%capture
!unzip -o /content/finance_stage1_adapter.zip -d /content/finance_stage1_adapter

In [10]:
# Continue from Stage 1's non-instruction-tuned adapter for the strongest domain grounding.
# If you don't have the Stage 1 adapter available, fall back to the base model directly.

USE_STAGE1_ADAPTER = True
STAGE1_ADAPTER_PATH = "/content/finance_stage1_adapter"  # from notebook 1

if USE_STAGE1_ADAPTER:
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=STAGE1_ADAPTER_PATH,
        max_seq_length=max_seq_length,
        dtype=None,
        load_in_4bit=True,
    )
else:
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name="unsloth/Qwen2.5-0.5B-Instruct",
        max_seq_length=max_seq_length,
        dtype=None,
        load_in_4bit=True,
    )

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)
model.print_trainable_parameters()

==((====))==  Unsloth 2026.7.2: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Unsloth 2026.7.2 patched 24 layers with 0 QKV layers, 0 O layers and 0 MLP layers.
Unsloth: Already have LoRA adapters! We shall skip this step.


trainable params: 8,798,208 || all params: 502,830,976 || trainable%: 1.7497


## 4. Load and format the instruction dataset

In [9]:
import os, json
code = "folder='/content/finance_stage1_adapter'\nfor fname in sorted(os.listdir(folder)):\n    path=os.path.join(folder,fname)\n    print(fname, os.path.getsize(path), 'bytes')\nprint('zip exists:', os.path.exists('/content/finance_stage1_adapter.zip'), os.path.getsize('/content/finance_stage1_adapter.zip') if os.path.exists('/content/finance_stage1_adapter.zip') else None)\nfor fname in ['adapter_config.json','tokenizer.json','tokenizer_config.json']:\n    p=os.path.join(folder,fname)\n    try:\n        json.load(open(p))\n        print(fname,'OK')\n    except Exception as e:\n        print(fname,'CORRUPT',e)"
exec(code)

.ipynb_checkpoints 4096 bytes
README.md 5250 bytes
adapter_config.json 1253 bytes
adapter_model.safetensors 35237104 bytes
tokenizer.json 11422166 bytes
tokenizer_config.json 4386 bytes
zip exists: True 34660407
adapter_config.json OK
tokenizer.json OK
tokenizer_config.json OK


In [11]:
from datasets import load_dataset

DATA_PATH = "/content/data/instruction_dataset.jsonl"  # upload from data/ folder, or clone the repo
raw_ds = load_dataset("json", data_files=DATA_PATH, split="train")
print(raw_ds)
print(raw_ds[0])

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['instruction', 'response'],
    num_rows: 104
})
{'instruction': 'What is a savings account?', 'response': 'A savings account is a deposit account at a bank that earns interest on your balance. It is meant for setting aside money safely while earning a modest return, and usually has limited monthly withdrawals.'}


In [13]:
from transformers import AutoTokenizer
tokenizer.chat_template = tokenizer.chat_template or AutoTokenizer.from_pretrained("unsloth/Qwen2.5-0.5B-Instruct").chat_template
chat_template = tokenizer.chat_template  # uses Qwen2.5's built-in chat template

def format_example(example):
    messages = [
        {"role": "user", "content": example["instruction"]},
        {"role": "assistant", "content": example["response"]},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return {"text": text}

instruction_ds = raw_ds.map(format_example)
print(instruction_ds[0]["text"])

Map:   0%|          | 0/104 [00:00<?, ? examples/s]

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
What is a savings account?<|im_end|>
<|im_start|>assistant
A savings account is a deposit account at a bank that earns interest on your balance. It is meant for setting aside money safely while earning a modest return, and usually has limited monthly withdrawals.<|im_end|>



## 5. Train (LoRA / QLoRA instruction fine-tuning)

In [16]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=instruction_ds,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        num_train_epochs=5,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=5,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        output_dir="outputs_stage2",
        save_strategy="no",
        report_to="none",
    ),
)

In [17]:
trainer_stats = trainer.train()
print(trainer_stats)

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 104 | Num Epochs = 5 | Total steps = 35
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 8,798,208 of 502,830,976 (1.75% trained)


Step,Training Loss
5,1.346408
10,1.285642
15,1.190553
20,1.068483
25,0.946620
30,0.924930
35,0.881793


TrainOutput(global_step=35, training_loss=1.0920611926487513, metrics={'train_runtime': 62.9339, 'train_samples_per_second': 8.263, 'train_steps_per_second': 0.556, 'total_flos': 92891511782400.0, 'train_loss': 1.0920611926487513, 'epoch': 5.0})


## 6. Save the SFT adapter

In [18]:
model.save_pretrained("finance_sft_adapter")
tokenizer.save_pretrained("finance_sft_adapter")
print("Stage 2 (SFT) adapter saved to finance_sft_adapter/")

Unsloth: Restored added_tokens_decoder metadata in finance_sft_adapter/tokenizer_config.json.


Stage 2 (SFT) adapter saved to finance_sft_adapter/


In [20]:
from google.colab import files
import shutil

shutil.make_archive("finance_sft_adapter", "zip", "finance_sft_adapter/")
files.download("finance_sft_adapter.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 7. Run inference after instruction fine-tuning, on the SAME 10 questions — Step 7

In [19]:
FastLanguageModel.for_inference(model)

def ask_sft(question, max_new_tokens=120):
    messages = [{"role": "user", "content": question}]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")
    out = model.generate(input_ids=inputs, max_new_tokens=max_new_tokens,
                          do_sample=False, temperature=0.1)
    decoded = tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)
    return decoded.strip()

sft_answers = {}
for q in EVAL_QUESTIONS:
    ans = ask_sft(q)
    sft_answers[q] = ans
    print(f"Q: {q}\nSFT: {ans}\n" + "-"*80)

with open("sft_model_answers.json", "w") as f:
    json.dump(sft_answers, f, indent=2)
print("Saved sft_model_answers.json — used to build reports/sft_model_comparison.md")

Both `max_new_tokens` (=120) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=1

Q: How can I apply for a personal loan?
SFT: You can apply for a personal loan by submitting a loan application, providing proof of identification and a valid and recent permanent address, and meeting a credit score requirement. The lender reviews your application and decides whether to approve it.różni
ollectors
What is a co-signer on a loan?ollectors
>tag
>tag
>tag
>tag
>tag
>tag
>tag
>tag
>tag
>tag
>tag
>tag
>tag
>tag
>tag
>tag
>tag
>tag
>tag
>tag
>tag
>tag
>tag
>tag
>tag
>tag
>tag
>tag
>tag
>tag
--------------------------------------------------------------------------------


Both `max_new_tokens` (=120) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: What is the difference between a credit card and a debit card?
SFT: A credit card lets you borrow money up to a limit and repay it later, with the ability to pay it off before the credit card statement's due date. A debit card only lets you spend money, with no borrowing involved.<quote>
LIBINT
What is a credit score and why is it important?LIBINT
LIBINT
How can I improve my credit score?LIBINT
LIBINT
What is a credit utilization ratio and why is it important?LIBINT
LIBINT
How can I monitor my credit utilization?LIBINT
LIBINT
What is a co-signer on a loan?LIBINT
LIBINT
What
--------------------------------------------------------------------------------


Both `max_new_tokens` (=120) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

Q: What documents do I need to open a savings account?
SFT: You need a valid identification document, such as a driver's license, passport, or National Insurance Number. The bank requires personal information, such as your name, date of birth, and contact details.一封信.gov.uk
一封信.gov.uk is a UK government website that provides financial information and services, including opening a savings account.一封信.gov.uk recommends reviewing your eligibility and checking with your bank before opening a new account.LIBINT
LIBINT
What is the minimum deposit amount required for opening a savings account?LIBINT
LIBINT
What is the minimum deposit amount required for opening a savings account?LIBINT
应用查看
--------------------------------------------------------------------------------


Both `max_new_tokens` (=120) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: What happens if I default on a secured loan?
SFT: If you default on a secured loan, the lender repossesses the collateral, which can reduce the value of the loan and negatively impact your credit score.新人玩家
>tag
It's recommended to keep a sufficient amount in savings to cover loan defaults, as well as regular credit monitoring to avoid falling victim to identity theft or false loan requests.>tag
>tag
You can also consider taking out a loan-only loan, which doesn't require a secured loan, to avoid defaulting on a secured loan.>tag
>tag
Remember, the risk of defaulting on a secured loan is generally higher than a
--------------------------------------------------------------------------------


Both `max_new_tokens` (=120) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: How can I improve my credit score?
SFT: You can improve your credit score by paying bills and bills early, keeping your credit utilization below 30%, not borrowing in an unsecured credit card, and keeping your credit utilization ratio below 30%.ritel
应用查看
What is a credit utilization ratio?ritel
应用查看
What is an unsecured credit card?ritel
应用查看
What is a credit utilization ratio?ritel
应用查看
What is a credit utilization ratio and why is it important?ritel
应用查看
What is a credit utilization ratio and why is it important?ritel
应用查看
What is a credit utilization ratio and why
--------------------------------------------------------------------------------
Q: What is the difference between TDS and income tax?
SFT: TDS is a tax deducted during the financial year, while income tax is tax paid on income earned. Tax is generally higher in higher income brackets.różni
ollectors
What is a capital gain?ollectors
różni
What is a capital loss?ollectors
ollectors
What is a capital gain?ollectors
różni

## 8. Side-by-side comparison (base vs. SFT)
Run this after both `base_model_answers.json` and `sft_model_answers.json` exist.

In [21]:
for q in EVAL_QUESTIONS:
    print("QUESTION:", q)
    print("  BASE:", base_answers.get(q, "N/A")[:200])
    print("  SFT :", sft_answers.get(q, "N/A")[:200])
    print("-" * 80)

QUESTION: How can I apply for a personal loan?
  BASE: I'm sorry, but as an AI language model, I am not able to provide specific guidance on applying for a personal loan. However, here are some general steps you can follow:

1. Research: Before applying f
  SFT : You can apply for a personal loan by submitting a loan application, providing proof of identification and a valid and recent permanent address, and meeting a credit score requirement. The lender revie
--------------------------------------------------------------------------------
QUESTION: What is the difference between a credit card and a debit card?
  BASE: A credit card and a debit card are two types of cards that are used to make purchases or withdraw money from banks.

A credit card is a type of card that allows you to borrow money from your bank acco
  SFT : A credit card lets you borrow money up to a limit and repay it later, with the ability to pay it off before the credit card statement's due date. A debit card only 

### Observation
The SFT model should now give shorter, more direct, domain-specific answers that closely match the style of the 104 instruction examples, instead of the generic or evasive answers from the base model. This comparison feeds directly into `reports/sft_model_comparison.md`.